# Notebook for visualizing the disk schemes in both cities

## Haha

In [209]:
import os
import sys

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder (e.g., 'masteroppgave') by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Find and append 'masteroppgave' to sys.path dynamically
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")


import folium
from folium.plugins import FeatureGroupSubGroup
import random
from schemes.lsh_disk import DiskLSH
from constants import  *
from utils.helpers import file_handler

Project root found: /cluster/home/thomanit/work/master/masteroppgave


## ROME

In [210]:
ROME_DATA_FOLDER = "../dataset/rome/output/"
PORTO_DATA_FOLDER = "../dataset/porto/output/"
CITY= "rome"
DIAMETER = 0.6
LAYERS = 4
DISKS =  250

if CITY == "rome":
    DiskLSH = DiskLSH(
        name="Disk1",
        min_lat = R_MIN_LAT,
        max_lat=R_MAX_LAT,
        min_lon=R_MIN_LON,
        max_lon=R_MAX_LON,
        disks=DISKS,
        layers=LAYERS,
        diameter=DIAMETER,
        meta_file="meta.txt",
        data_path="data",
    )
    print("running rome")
    print(DiskLSH)
    
if CITY == "porto":
    DiskLSH = DiskLSH(
        name="Disk2",
        min_lat = P_MIN_LAT,
        max_lat=P_MAX_LAT,
        min_lon=P_MIN_LON,
        max_lon=P_MAX_LON,
        disks=DISKS,
        layers=LAYERS,
        diameter=DIAMETER,
        meta_file="meta.txt",
        data_path="data",
    )

    print("running porto")
    print(DiskLSH)
    

running rome
Disk-scheme: Disk1 
Covering: (5.559754011676299, 7.451072531046803) km 
Diameter: 0.6 km
Layers: 4 



### Visualize disk scheme on map

In [211]:
import folium
from folium.plugins import FeatureGroupSubGroup
def visualize_disks_with_boundary(disk_lsh, width=800, height=600):
    """
    Visualizes the disks of the DiskLSH object using Folium and adds a bounding box.

    Parameters:
    - disk_lsh (DiskLSH): An instance of the DiskLSH class.
    - width (int): Width of the map in pixels (default: 800)
    - height (int): Height of the map in pixels (default: 600)

    Returns:
    - A Folium map object.
    """

    # Define center of the map (average lat/lon)
    center_lat = (disk_lsh.min_lat + disk_lsh.max_lat) / 2
    center_lon = (disk_lsh.min_lon + disk_lsh.max_lon) / 2

    # Initialize folium map
    map_disks = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="OpenStreetMap", width=width, height=height)

    # Define colors for different layers
    layer_colors = ["red", "blue", "green", "purple", "orange"]

    # Create a base layer group
    base_layer = folium.FeatureGroup(name="Base Map").add_to(map_disks)

    # Add bounding box (dataset boundary)
    folium.Rectangle(
        bounds=[(disk_lsh.min_lat, disk_lsh.min_lon), (disk_lsh.max_lat, disk_lsh.max_lon)],
        color="black",
        weight=2,
        fill=True,
        fill_opacity=0.1,
        popup="Bounding Box"
    ).add_to(base_layer)

    # Iterate over each layer in the disk structure
    for layer_index, (layer, disks) in enumerate(disk_lsh.disks.items()):
        color = layer_colors[layer_index % len(layer_colors)]  # Cycle colors

        # Create a subgroup for each layer
        layer_group = FeatureGroupSubGroup(base_layer, name=f"Layer {layer_index + 1}")
        map_disks.add_child(layer_group)

        # Plot disks as circles
        for disk in disks:
            lat, lon = disk  # Disk center
            folium.Circle(
                location=[lat, lon],
                radius=disk_lsh.diameter * 500,  # Convert km to meters
                color=color,
                fill=True,
                fill_opacity=0.4,
                popup=f"Layer {layer_index + 1}\nDisk: ({lat:.5f}, {lon:.5f})",
            ).add_to(layer_group)

    # Add layer control to toggle between layers
    folium.LayerControl(collapsed=False).add_to(map_disks)

    return map_disks
disk_map = visualize_disks_with_boundary(DiskLSH, width=1000, height=800)
disk_map

### Add trajectory


In [212]:
if CITY == "rome":
    trajectories = ["R_ABA.txt"]
    trajs_2 = file_handler.load_trajectory_files(trajectories, ROME_DATA_FOLDER)
    hash_num = DiskLSH._create_trajectory_hash_with_KD_tree_numerical(trajs_2["R_ABA"])
    hash_letter = DiskLSH._create_trajectory_hash_with_KD_tree(trajs_2["R_ABA"])
    print(hash_num)
    print(hash_letter) 

if CITY == "porto":
    trajectories = ["P_ABA.txt"]
    trajs_2 = file_handler.load_trajectory_files(["P_ABA.txt"], PORTO_DATA_FOLDER)
    hash_num = DiskLSH._create_trajectory_hash_with_KD_tree_numerical(trajs_2["P_ABA"])
    hash_letter = DiskLSH._create_trajectory_hash_with_KD_tree(trajs_2["P_ABA"])
    print(hash_num)
    print(hash_letter)  


[[array([41.91857325, 12.49188239]), array([41.91616814, 12.49860219]), array([41.911735  , 12.49508439]), array([41.90854381, 12.49059054]), array([41.90511357, 12.49035899]), array([41.90586217, 12.48866158]), array([41.90551228, 12.4919291 ]), array([41.9060136 , 12.49444312]), array([41.89985187, 12.48807297])], [array([41.91237216, 12.49236228]), array([41.91098622, 12.49004322]), array([41.9083632 , 12.48960289]), array([41.90788606, 12.48795658]), array([41.9107627 , 12.48768899]), array([41.90711419, 12.49100743]), array([41.90183216, 12.49514075])], [array([41.91802882, 12.48921351]), array([41.91744893, 12.492795  ]), array([41.91807391, 12.4912889 ]), array([41.91743629, 12.4958213 ]), array([41.91802882, 12.48921351]), array([41.91744893, 12.492795  ]), array([41.91743629, 12.4958213 ]), array([41.91812886, 12.49728549]), array([41.91744893, 12.492795  ]), array([41.91744893, 12.492795  ]), array([41.91173152, 12.49134465]), array([41.90943485, 12.48838873]), array([41.9105

### Visualize disks, trajectory and hashed trajectory on map

In [213]:
import folium
from folium.plugins import FeatureGroupSubGroup
import random

def visualize_disks_with_trajectory(disk_lsh, trajectory, hashed_trajectory):
    """
    Visualizes the disks of the DiskLSH object using Folium.
    - Trajectory is displayed as a blue polyline.
    - Hashed points are marked with colored circles.

    Parameters:
    - disk_lsh (DiskLSH): An instance of the DiskLSH class.
    - trajectory (list): List of (lat, lon) coordinates representing a single trajectory.
    - hashed_trajectory (list of lists): Hashed trajectory representation per layer.

    Returns:
    - A Folium map object.
    """

    # Define center of the map (average lat/lon)
    center_lat = (disk_lsh.min_lat + disk_lsh.max_lat) / 2
    center_lon = (disk_lsh.min_lon + disk_lsh.max_lon) / 2

    # Initialize folium map
    map_disks = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="OpenStreetMap")

    # Define colors for different disk layers
    layer_colors = ["red", "blue", "green", "purple", "orange"]

    # Create a base layer group
    base_layer = folium.FeatureGroup(name="Base Map").add_to(map_disks)

    # Add bounding box (dataset boundary)
    folium.Rectangle(
        bounds=[(disk_lsh.min_lat, disk_lsh.min_lon), (disk_lsh.max_lat, disk_lsh.max_lon)],
        color="black",
        weight=2,
        fill=True,
        fill_opacity=0.1,
        popup="Bounding Box"
    ).add_to(base_layer)

    # Iterate over each layer in the disk structure
    for layer_index, (layer, disks) in enumerate(disk_lsh.disks.items()):
        color = layer_colors[layer_index % len(layer_colors)]  # Cycle colors

        # Create a subgroup for each layer
        layer_group = FeatureGroupSubGroup(base_layer, name=f"Layer {layer_index + 1}")
        map_disks.add_child(layer_group)

        # Plot disks as circles
        for disk in disks:
            lat, lon = disk  # Disk center
            folium.Circle(
                location=[lat, lon],
                radius=disk_lsh.diameter * 500,  # Convert km to meters
                color=color,
                fill=True,
                fill_opacity=0.4,
                popup=f"Layer {layer_index + 1}\nDisk: ({lat:.5f}, {lon:.5f})",
            ).add_to(layer_group)

    # Plot the trajectory as a blue polyline
    folium.PolyLine(
        trajectory,
        color="blue",
        weight=2,
        opacity=1,
        popup="Original Trajectory"
    ).add_to(base_layer)

    # Add markers for each point along the trajectory
    for lat, lon in trajectory:
        folium.CircleMarker(
            location=(lat, lon),
            radius=3,  # Small marker
            color="black",
            fill=True,
            fill_opacity=1,
            popup=f"Point: ({lat:.5f}, {lon:.5f})"
        ).add_to(base_layer)

    # Plot hashed trajectory points as markers with circles
    hashed_layer_colors = ["red", "blue", "green", "purple", "orange"]  # Colors for different layers

    for layer_index, layer_points in enumerate(hashed_trajectory):
        layer_color = hashed_layer_colors[layer_index % len(hashed_layer_colors)]  # Assign color per layer

        for lat, lon in layer_points:
            # Add a small circle around the hashed point
            folium.Circle(
                location=(lat, lon),
                radius=20,  # Small circle
                color=layer_color,
                fill=True,
                fill_opacity=0.4,
                popup=f"Layer {layer_index + 1} Hashed Point"
            ).add_to(base_layer)

            # Add a marker on top of the hashed point
            folium.Marker(
                location=(lat, lon),
                icon=folium.Icon(color=layer_color, icon="info-sign"),
                popup=f"Hashed Point ({lat:.5f}, {lon:.5f}) - Layer {layer_index + 1}"
            ).add_to(base_layer)

    # Add layer control to toggle between layers
    folium.LayerControl(collapsed=False).add_to(map_disks)

    return map_disks


disk_map = visualize_disks_with_trajectory(DiskLSH, trajs_2["R_ABA"], hash_num)
disk_map.save("output/rome/rome_disk_with_hashed_trajectory.html")

print(f"Saved disk visualization with hashed trajectory to output/rome/rome_disk_with_hashed_trajectory.html")


Saved disk visualization with hashed trajectory to output/rome/rome_disk_with_hashed_trajectory.html
